In [34]:
from pathlib import Path

import matplotlib.pyplot as plt
import requests
import torch
import torchmetrics
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

In [2]:
p = Path(".")
[x for x in p.iterdir()]

[PosixPath('uv.lock'),
 PosixPath('pyproject.toml'),
 PosixPath('datasets'),
 PosixPath('utils'),
 PosixPath('README.md'),
 PosixPath('.gitignore'),
 PosixPath('RNN_with_Attention.ipynb'),
 PosixPath('.venv'),
 PosixPath('.python-version'),
 PosixPath('.git'),
 PosixPath('.vscode')]

In [25]:
device = "mps"

In [3]:
def download_text(text_path, url_to_download):
    path = Path(text_path)
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = url_to_download
        print(url)
        r = requests.get(url)
        with open(path, "wb") as f:
            f.write(r.content)
    return path.read_text()


shakespeare_text = download_text(
    text_path="datasets/shakespeare/shakespeare.txt",
    url_to_download="https://homl.info/shakespeare",
)


In [4]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [5]:
vocab = sorted(set(shakespeare_text.lower()))
"".join(vocab)

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [6]:
char_to_id = {char: index for index, char in enumerate(vocab)}
id_to_char = {index: char for index, char in enumerate(vocab)}

In [7]:
char_to_id["b"]

14

In [8]:
id_to_char[12]

'?'

In [9]:
def encode_text(text):
    return torch.tensor([char_to_id[x] for x in text.lower()])


def decode_text(char_ids):
    return "".join([id_to_char[char_id.item()] for char_id in char_ids])

In [10]:
encoder = encode_text("Cosminius")
encoder

tensor([15, 27, 31, 25, 21, 26, 21, 33, 31])

In [11]:
decode_text(encoder)

'cosminius'

In [12]:
  class CharDataset(Dataset):
    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length

    def __len__(self):
        return len(self.encoded_text) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        end = idx + self.window_length
        window = self.encoded_text[idx:end]
        target = self.encoded_text[idx + 1 : end + 1]
        return window, target

In [13]:
window_length = 50
batch_size = 512

train_set = CharDataset(shakespeare_text[:1_000_000], window_length)
valid_set = CharDataset(shakespeare_text[1_000_000:1_060_000], window_length)
test_set = CharDataset(shakespeare_text[1_060_000:], window_length)

In [14]:
train_set[0]

(tensor([18, 21, 30, 31, 32,  1, 15, 21, 32, 21, 38, 17, 26, 10,  0, 14, 17, 18,
         27, 30, 17,  1, 35, 17,  1, 28, 30, 27, 15, 17, 17, 16,  1, 13, 26, 37,
          1, 18, 33, 30, 32, 20, 17, 30,  6,  1, 20, 17, 13, 30]),
 tensor([21, 30, 31, 32,  1, 15, 21, 32, 21, 38, 17, 26, 10,  0, 14, 17, 18, 27,
         30, 17,  1, 35, 17,  1, 28, 30, 27, 15, 17, 17, 16,  1, 13, 26, 37,  1,
         18, 33, 30, 32, 20, 17, 30,  6,  1, 20, 17, 13, 30,  1]))

In [15]:
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

In [16]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def train(
    model,
    optimizer,
    loss_fn,
    metric,
    train_loader,
    valid_loader,
    n_epochs,
    patience=2,
    factor=0.5,
    epoch_callback=None,
):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor
    )
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss / (index + 1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(
            f"\rEpoch {epoch + 1}/{n_epochs},                      "
            f"train loss: {history['train_losses'][-1]:.4f}, "
            f"train metric: {history['train_metrics'][-1]:.2%}, "
            f"valid metric: {history['valid_metrics'][-1]:.2%}"
        )
    return history

In [17]:
to_be_dataset = CharDataset("To be or not to be", window_length=10)
for x, y in to_be_dataset:
    print(f"x={x}, y={y}")
    print(f"    decoded: x={decode_text(x)!r}, y={decode_text(y)!r}")

x=tensor([32, 27,  1, 14, 17,  1, 27, 30,  1, 26]), y=tensor([27,  1, 14, 17,  1, 27, 30,  1, 26, 27])
    decoded: x='to be or n', y='o be or no'
x=tensor([27,  1, 14, 17,  1, 27, 30,  1, 26, 27]), y=tensor([ 1, 14, 17,  1, 27, 30,  1, 26, 27, 32])
    decoded: x='o be or no', y=' be or not'
x=tensor([ 1, 14, 17,  1, 27, 30,  1, 26, 27, 32]), y=tensor([14, 17,  1, 27, 30,  1, 26, 27, 32,  1])
    decoded: x=' be or not', y='be or not '
x=tensor([14, 17,  1, 27, 30,  1, 26, 27, 32,  1]), y=tensor([17,  1, 27, 30,  1, 26, 27, 32,  1, 32])
    decoded: x='be or not ', y='e or not t'
x=tensor([17,  1, 27, 30,  1, 26, 27, 32,  1, 32]), y=tensor([ 1, 27, 30,  1, 26, 27, 32,  1, 32, 27])
    decoded: x='e or not t', y=' or not to'
x=tensor([ 1, 27, 30,  1, 26, 27, 32,  1, 32, 27]), y=tensor([27, 30,  1, 26, 27, 32,  1, 32, 27,  1])
    decoded: x=' or not to', y='or not to '
x=tensor([27, 30,  1, 26, 27, 32,  1, 32, 27,  1]), y=tensor([30,  1, 26, 27, 32,  1, 32, 27,  1, 14])
    decoded: x=

In [18]:
import torch.nn as nn

torch.manual_seed(42)
embed = nn.Embedding(5, 3)
embed(torch.tensor([[3, 2], [0, 2]]))


tensor([[[ 0.2674,  0.5349,  0.8094],
         [ 2.2082, -0.6380,  0.4617]],

        [[ 0.3367,  0.1288,  0.2345],
         [ 2.2082, -0.6380,  0.4617]]], grad_fn=<EmbeddingBackward0>)

In [19]:
embed(torch.tensor([3]))

tensor([[0.2674, 0.5349, 0.8094]], grad_fn=<EmbeddingBackward0>)

In [23]:
linear_layer = nn.Linear(5, 3, bias=False, dtype=torch.float32)
one_hot_encoding = torch.tensor([0, 0, 1, 0, 1], dtype=torch.float32)
linear_layer(one_hot_encoding)

tensor([0.0667, 0.5273, 0.0223], grad_fn=<SqueezeBackward4>)

In [26]:
class ShakespeareModel(nn.Module):
    def __init__(
        self, vocab_size, n_layer=2, embed_dim=10, hidden_dim=128, dropout=0.1
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layer, batch_first=True, dropout=dropout
        )
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, _states = self.gru(embeddings)
        return self.output(outputs).permute(0, 2, 1)


torch.manual_seed(42)
model = ShakespeareModel(len(vocab)).to(device)

# The permutatio nat the end is because CrossEntropy and accuracy expect the class dimension to be second dimension not last one

In [28]:
n_epochs = 20
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=len(vocab)).to(device)
history = train(
    model, optimizer, xentropy, accuracy, train_loader, valid_loader, n_epochs
)

Epoch 1/20,                      train loss: 1.6048, train metric: 51.26%, valid metric: 51.46%
Epoch 2/20,                      train loss: 1.3857, train metric: 56.67%, valid metric: 53.03%
Epoch 3/20,                      train loss: 1.3559, train metric: 57.42%, valid metric: 53.96%
Epoch 4/20,                      train loss: 1.3415, train metric: 57.80%, valid metric: 53.60%
Epoch 5/20,                      train loss: 1.3330, train metric: 58.02%, valid metric: 53.86%
Epoch 6/20,                      train loss: 1.3271, train metric: 58.17%, valid metric: 54.08%
Epoch 7/20,                      train loss: 1.3229, train metric: 58.28%, valid metric: 54.05%
Epoch 8/20,                      train loss: 1.3198, train metric: 58.34%, valid metric: 54.34%
Epoch 9/20,                      train loss: 1.3171, train metric: 58.41%, valid metric: 54.36%
Epoch 10/20,                      train loss: 1.3151, train metric: 58.46%, valid metric: 53.92%
Epoch 11/20,                      train

In [29]:
torch.save(model.state_dict(), "myshakespeare_model.pt")

In [30]:
model.eval()
text = "to be or not to b"
encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
with torch.no_grad():
    Y_logits = model(encoded_text)
    predicted_char_id = Y_logits[0, :, -1].argmax().item()
    predicted_char = id_to_char[predicted_char_id]

In [31]:
predicted_char

'e'

In [33]:
torch.manual_seed(42)
probs = torch.tensor([[0.5, 0.4, 0.1]])
samples = torch.multinomial(probs, replacement=True, num_samples=8)
samples

tensor([[0, 0, 0, 0, 1, 0, 2, 2]])

In [35]:
def next_char(model, text, temperature=1):
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()
    return id_to_char[predicted_char_id]

In [36]:
def extend_text(model, text, n_chars=80, temperature=1):
    for _ in range(n_chars):
        text += next_char(model, text, temperature)
    return text

In [37]:
print(extend_text(model, text="To be or not to b", temperature=0.01))

To be or not to be the state,
that i have stand the state and state and commandment that is the s


In [38]:
print(extend_text(model, text="To be or not to b", temperature=0.4))

To be or not to be content.

coriolanus:
the strokes to his power,
and the services me on the com


In [40]:
print(extend_text(model, text="To be or not to b", temperature=1.5))

To be or not to bear.
tull-peating ladien here
both his shame, where'as we and bigger wretch.
tho


In [41]:
print(extend_text(model, text="To be or not to b", temperature=100))

To be or not to bq,z'tayse-qvhxj,t!apgd?vp
:l?&xk.t;c u!gpcyg&g,j$u-fshef&nely-k-c'hu!mk?hejc:bmj


In [55]:
# Training a stateful RNN
class StatefulShakespeareModel(nn.Module):
    def __init__(
        self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128, dropout=0.1
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(
            embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.output = nn.Linear(hidden_dim, vocab_size)
        self.hidden_states = None

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, hidden_states = self.gru(embeddings, self.hidden_states)
        self.hidden_states = hidden_states.detach()
        return self.output(outputs).permute(0, 2, 1)

In [56]:
class StatefulCharDataset(Dataset):
    def __init__(self, text, window_length, batch_size):
        self.encoded_text = encode_text(text)
        self.window_length = window_length
        self.batch_size = batch_size
        n_consecutive_windows = (len(self.encoded_text) - 1) // window_length
        n_windows_per_slot = n_consecutive_windows // batch_size
        self.length = n_windows_per_slot * batch_size
        self.spacing = n_windows_per_slot * window_length

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        start = (idx % self.batch_size) * self.spacing + (
            idx // self.batch_size
        ) * self.window_length
        end = start + self.window_length
        window = self.encoded_text[start:end]
        target = self.encoded_text[start + 1 : end + 1]
        return window, target

In [57]:
batch_size = 128
stateful_train_set = StatefulCharDataset(
    shakespeare_text[:1_000_000], window_length, batch_size
)
stateful_train_loader = DataLoader(
    stateful_train_set, batch_size=batch_size, drop_last=True
)
stateful_valid_set = StatefulCharDataset(
    shakespeare_text[1_000_000:1_060_000], window_length, batch_size
)
stateful_valid_loader = DataLoader(
    stateful_valid_set, batch_size=batch_size, drop_last=True
)
stateful_test_set = StatefulCharDataset(
    shakespeare_text[1_060_000:], window_length, batch_size
)
stateful_test_loader = DataLoader(
    stateful_test_set, batch_size=batch_size, drop_last=True
)


In [58]:
torch.manual_seed(42)

stateful_model = StatefulShakespeareModel(len(vocab)).to(device)

n_epochs = 10
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(stateful_model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=len(vocab)).to(device)


def reset_hidden_states(model, epoch):
    model.hidden_states = None


history = train(
    stateful_model,
    optimizer,
    xentropy,
    accuracy,
    stateful_train_loader,
    stateful_valid_loader,
    n_epochs,
    epoch_callback=reset_hidden_states,
)

Epoch 1/10,                      train loss: 2.4744, train metric: 29.82%, valid metric: 39.05%
Epoch 2/10,                      train loss: 1.8777, train metric: 44.47%, valid metric: 44.65%
Epoch 3/10,                      train loss: 1.6918, train metric: 49.37%, valid metric: 47.77%
Epoch 4/10,                      train loss: 1.5958, train metric: 51.79%, valid metric: 49.78%
Epoch 5/10,                      train loss: 1.5394, train metric: 53.21%, valid metric: 50.94%
Epoch 6/10,                      train loss: 1.5016, train metric: 54.19%, valid metric: 51.53%
Epoch 7/10,                      train loss: 1.4734, train metric: 54.88%, valid metric: 52.33%
Epoch 8/10,                      train loss: 1.4523, train metric: 55.40%, valid metric: 52.78%
Epoch 9/10,                      train loss: 1.4345, train metric: 55.85%, valid metric: 53.17%
Epoch 10/10,                      train loss: 1.4201, train metric: 56.23%, valid metric: 53.65%


In [59]:
torch.save(stateful_model.state_dict(), "my_stateful_shakespeare_model.pt")

In [62]:
def extend_text_with_stateful_rnn(model, text, n_chars=80, temperature=1):
    model.hidden_states = None
    rnn_input = text
    for _ in range(n_chars):
        char = next_char(model, rnn_input, temperature)
        text += char
        rnn_input = char
    return text + "..."

In [ ]:
torch.manual_seed(42)
stateful_model.eval()
print(
    extend_text_with_stateful_rnn(stateful_model, "To be or not to b", temperature=0.1)
)

To be or not to be the people.

duke vincentio:
the son that the duke of the daughter the duke.

...


In [ ]:
print(
    extend_text_with_stateful_rnn(stateful_model, "To be or not to b", temperature=0.4)
)

To be or not to be call
the for you have here and the land and therefore
and when the end the con...


In [ ]:
print(extend_text_with_stateful_rnn(stateful_model, "To be or not to b", temperature=1))

To be or not to bears
to see a lately could the point-bid thee where
as we and by tell me?

warwi...
